In [1]:
import pandas as pd
import requests
import numpy as np
from bs4 import BeautifulSoup
import yfinance
import os
import pygsheets
from decimal import Decimal, ROUND_HALF_UP
import locale
from datetime import datetime, timedelta
import json
import datawrapper
import time
import io
#https://github.com/patrickloeber/python-github-action-template?tab=readme-ov-file
# Set the locale to English (United States)
locale.setlocale(locale.LC_TIME, 'en_US.UTF-8')
#authorization
if "PYGSHEET" in os.environ:

    AUTH_DATA = os.environ["PYGSHEET"]
    whatever_json=json.loads(AUTH_DATA, strict=False)
    credentials = whatever_json

    gc = pygsheets.authorize(service_account_json=(json.dumps(whatever_json))
    )
else:

    gc = pygsheets.authorize(service_file='/Users/halukamaier-borst/Documents/trumptracker-459020-5b428916efa8.json')



In [2]:

# Get today's date
today = datetime.now()

# Get date 365 days ago
date_365_days_ago = today - timedelta(days=365)

# Format as string if needed
date_365_days_ago_str = date_365_days_ago.strftime("%Y-%m-%d")

print(date_365_days_ago_str)
today

2025-01-06


datetime.datetime(2026, 1, 6, 9, 17, 46, 103810)

In [3]:

shanghaicomposite = yfinance.download (tickers = "000001.SS", start = date_365_days_ago_str, 
                              interval = "1d")
shanghaicomposite

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,000001.SS,000001.SS,000001.SS,000001.SS,000001.SS
Date,,,,,
2025-01-06,3206.923096,3219.488037,3185.462891,3209.782959,431000
2025-01-07,3229.644043,3230.853027,3190.460938,3203.306885,409700
2025-01-08,3230.167969,3246.291016,3175.725098,3218.857910,472900
2025-01-09,3211.393066,3228.972900,3205.909912,3220.720947,382900
2025-01-10,3168.523926,3220.106934,3168.523926,3211.707031,403700
...,...,...,...,...,...
2025-12-29,3965.280029,3983.979004,3956.946045,3964.650879,563700
2025-12-30,3965.115967,3979.989990,3947.418945,3947.871094,525900


In [4]:

dax40 = yfinance.download (tickers = "^GDAXI", start = date_365_days_ago_str,
                              interval = "1d")
dax40

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GDAXI,^GDAXI,^GDAXI,^GDAXI,^GDAXI
Date,,,,,
2025-01-06,20216.189453,20216.189453,19921.119141,19982.660156,70784900
2025-01-07,20340.570312,20391.169922,20152.699219,20206.039062,62020000
2025-01-08,20329.939453,20480.490234,20242.560547,20322.169922,62954900
2025-01-09,20317.099609,20360.380859,20245.960938,20284.199219,50177100
2025-01-10,20214.789062,20400.519531,20187.970703,20316.550781,76245500
...,...,...,...,...,...
2025-12-29,24351.119141,24389.150391,24253.730469,24370.939453,35372300
2025-12-30,24490.410156,24527.939453,24328.419922,24343.189453,23179300


In [5]:
sandp500 = yfinance.download (tickers = "^GSPC", start = date_365_days_ago_str, 
                              interval = "1d")
sandp500

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2025-01-06,5975.379883,6021.040039,5960.009766,5982.810059,4940120000
2025-01-07,5909.029785,6000.680176,5890.680176,5993.259766,4517330000
2025-01-08,5918.250000,5927.890137,5874.779785,5910.660156,4441740000
2025-01-10,5827.040039,5890.350098,5807.779785,5890.350098,4751930000
2025-01-13,5836.220215,5838.609863,5773.310059,5782.020020,4421200000
...,...,...,...,...,...
2025-12-29,6905.740234,6920.209961,6888.759766,6903.600098,3541750000
2025-12-30,6896.240234,6913.250000,6893.470215,6900.439941,3309930000


In [6]:
def reduce_index(df,indexname):
    index_red=pd.DataFrame()
    df["Date"]=df.index
    index_red["Date"]=df["Date"]

    index_red[indexname]=df[df.columns[0]]

    index_red=index_red.reset_index(drop=True)
    return index_red


In [7]:
sandp500_red=reduce_index(sandp500,"S&P500")
dax40_red=reduce_index(dax40,"DAX40")
shanghaicomposite_red=reduce_index(shanghaicomposite,"SHANGHAI")
shanghaicomposite_red

,Date,SHANGHAI
0,2025-01-06,3206.923096
1,2025-01-07,3229.644043
2,2025-01-08,3230.167969
3,2025-01-09,3211.393066
4,2025-01-10,3168.523926
...,...,...
238,2025-12-29,3965.280029
239,2025-12-30,3965.115967
240,2025-12-31,3968.840088
241,2026-01-05,4023.416992


In [8]:
sandp500_dax40=pd.merge(sandp500_red, dax40_red, on="Date", how="outer")
sandp500_dax40=sandp500_dax40.fillna(method="ffill")
sandp500_dax40

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_35927/2964685221.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sandp500_dax40=sandp500_dax40.fillna(method="ffill")


,Date,S&P500,DAX40
0,2025-01-06,5975.379883,20216.189453
1,2025-01-07,5909.029785,20340.570312
2,2025-01-08,5918.250000,20329.939453
3,2025-01-09,5918.250000,20317.099609
4,2025-01-10,5827.040039,20214.789062
...,...,...,...
254,2025-12-30,6896.240234,24490.410156
255,2025-12-31,6845.500000,24490.410156
256,2026-01-02,6858.470215,24539.339844
257,2026-01-05,6902.049805,24868.689453


In [9]:
sandp500_dax40_shanghai=pd.merge(sandp500_dax40, shanghaicomposite_red, on="Date", how="outer")
sandp500_dax40_shanghai=sandp500_dax40_shanghai.fillna(method="ffill")
sandp500_dax40_shanghai

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_35927/2476299639.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sandp500_dax40_shanghai=sandp500_dax40_shanghai.fillna(method="ffill")


,Date,S&P500,DAX40,SHANGHAI
0,2025-01-06,5975.379883,20216.189453,3206.923096
1,2025-01-07,5909.029785,20340.570312,3229.644043
2,2025-01-08,5918.250000,20329.939453,3230.167969
3,2025-01-09,5918.250000,20317.099609,3211.393066
4,2025-01-10,5827.040039,20214.789062,3168.523926
...,...,...,...,...
256,2025-12-30,6896.240234,24490.410156,3965.115967
257,2025-12-31,6845.500000,24490.410156,3968.840088
258,2026-01-02,6858.470215,24539.339844,3968.840088
259,2026-01-05,6902.049805,24868.689453,4023.416992


In [10]:
now = datetime.now()
today=now.strftime("%Y-%m-%d")
since_election=sandp500_dax40_shanghai[sandp500_dax40_shanghai["Date"]>=date_365_days_ago_str]
since_election_without_today=since_election[since_election["Date"]!=today]
since_election_without_today

,Date,S&P500,DAX40,SHANGHAI
0,2025-01-06,5975.379883,20216.189453,3206.923096
1,2025-01-07,5909.029785,20340.570312,3229.644043
2,2025-01-08,5918.250000,20329.939453,3230.167969
3,2025-01-09,5918.250000,20317.099609,3211.393066
4,2025-01-10,5827.040039,20214.789062,3168.523926
...,...,...,...,...
255,2025-12-29,6905.740234,24351.119141,3965.280029
256,2025-12-30,6896.240234,24490.410156,3965.115967
257,2025-12-31,6845.500000,24490.410156,3968.840088
258,2026-01-02,6858.470215,24539.339844,3968.840088


In [11]:
for col in since_election_without_today.columns[1:]:
    since_election_without_today[col]= since_election_without_today[col].apply(lambda x: Decimal(str(x)))

    # Example: Perform precise rounding on the Decimal column
    since_election_without_today[col]= since_election_without_today[col].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
since_election_without_today

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_35927/3670032238.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  since_election_without_today[col]= since_election_without_today[col].apply(lambda x: Decimal(str(x)))
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_35927/3670032238.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  since_election_without_today[col]= since_election_without_today[col].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))


,Date,S&P500,DAX40,SHANGHAI
0,2025-01-06,5975,20216,3207
1,2025-01-07,5909,20341,3230
2,2025-01-08,5918,20330,3230
3,2025-01-09,5918,20317,3211
4,2025-01-10,5827,20215,3169
...,...,...,...,...
255,2025-12-29,6906,24351,3965
256,2025-12-30,6896,24490,3965
257,2025-12-31,6846,24490,3969
258,2026-01-02,6858,24539,3969


In [12]:
since_election_without_today[since_election_without_today["Date"]=="2025-05-01"]

,Date,S&P500,DAX40,SHANGHAI
83,2025-05-01,5604,22497,3279


In [ ]:
def retrieve_from_investing(index):
    index_to_shortform_dict={"S&P500":"us-spx-500","DAX40":"germany-30",	"SHANGHAI":"shanghai-composite"}
    now = datetime.now()
    yesterday = now - timedelta(days=1)
    formatted_date = yesterday.strftime("%m/%d/%Y")
    index_shortform=index_to_shortform_dict[index]

    time.sleep(10)

    url=f"https://www.investing.com/indices/{index_shortform}-historical-data#:~:text=Daily-,Download,-{formatted_date}"
    print(url)
    response = requests.get(url)


    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content of the page
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the first table on the page
        table = soup.find('table', class_="freeze-column-w-1 w-full overflow-x-auto text-xs leading-4")

        # Extract rows from the table
        rows = table.find_all('tr')

        # Extract data from each row
        table_data = []
        for row in rows:
            cells = row.find_all(['td', 'th'])  # Get all cells (both <td> and <th>)
            cell_data = [cell.get_text(strip=True) for cell in cells]  # Extract text
            table_data.append(cell_data)

        # Return the extracted table data
        return(table_data)
    else:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")

In [14]:
df_to_check=since_election_without_today
columns_to_round = ['S&P500','DAX40','SHANGHAI']

# Apply Decimal rounding to the specified columns
df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

for index_name in since_election_without_today.columns[1:]:
    print(index_name)
    array=retrieve_from_investing(index_name)
    crosscheck_data=pd.DataFrame(array)
    crosscheck_data.replace("",np.nan,inplace=True)

    crosscheck_data=crosscheck_data.dropna(how="all")
    crosscheck_data[0]=pd.to_datetime(crosscheck_data[0], format="%b %d, %Y", errors="coerce")
    crosscheck_data_red=crosscheck_data[[0,1]]
    crosscheck_data_red=crosscheck_data_red.dropna(subset=[0])
    crosscheck_data_red.columns=["Date",index_name+"check"]
    crosscheck_data_red=crosscheck_data_red.sort_values(by="Date", ascending=True)
    crosscheck_data_red[index_name+"check"]=crosscheck_data_red[index_name+"check"].str.replace(",","")
    crosscheck_data_red[index_name+"check"]=crosscheck_data_red[index_name+"check"].astype(float)

    crosscheck_data_red[index_name+"check"]= crosscheck_data_red[index_name+"check"].apply(lambda x: Decimal(str(x)))

# Example: Perform precise rounding on the Decimal column
    crosscheck_data_red[index_name+"check"] = crosscheck_data_red[index_name+"check"].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
    print("crosscheckdata")
    print(crosscheck_data_red.tail(5))
    print("df_to_check")

    print(df_to_check.tail(5))

#    df_to_check=pd.merge(df_to_check,crosscheck_data_red, on="Date", how="outer", suffixes=["_raw","_crosscheck"])

    df_to_check=pd.merge(df_to_check,crosscheck_data_red, on="Date", how="outer", suffixes=["_raw","_crosscheck"])
df_to_check=df_to_check.fillna(method="ffill")
df_to_check


/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_85003/1710883695.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_85003/1710883695.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))


S&P500
https://www.investing.com/indices/us-spx-500-historical-data#:~:text=Daily-,Download,-12/10/2025


/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_85003/1710883695.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  crosscheck_data.replace("",np.nan,inplace=True)


crosscheckdata
        Date S&P500check
5 2025-12-03        6850
4 2025-12-04        6857
3 2025-12-05        6870
2 2025-12-08        6847
1 2025-12-09        6841
df_to_check
          Date S&P500  DAX40 SHANGHAI
255 2025-12-03   6850  23694     3878
256 2025-12-04   6857  23882     3876
257 2025-12-05   6870  24028     3903
258 2025-12-08   6847  24046     3924
259 2025-12-09   6841  24163     3910
DAX40
https://www.investing.com/indices/germany-30-historical-data#:~:text=Daily-,Download,-12/10/2025
crosscheckdata
        Date DAX40check
5 2025-12-04      23882
4 2025-12-05      24028
3 2025-12-08      24046
2 2025-12-09      24163
1 2025-12-10      24061
df_to_check
          Date S&P500  DAX40 SHANGHAI S&P500check
255 2025-12-03   6850  23694     3878        6850
256 2025-12-04   6857  23882     3876        6857
257 2025-12-05   6870  24028     3903        6870
258 2025-12-08   6847  24046     3924        6847
259 2025-12-09   6841  24163     3910        6841
SHANGHAI
https://www.

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_85003/1710883695.py:35: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_to_check=df_to_check.fillna(method="ffill")


,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck
0,2024-12-10,6035,20329,3423,NaN,NaN,NaN
1,2024-12-11,6084,20399,3432,NaN,NaN,NaN
2,2024-12-12,6051,20426,3462,NaN,NaN,NaN
3,2024-12-13,6051,20406,3392,NaN,NaN,NaN
4,2024-12-16,6074,20314,3386,NaN,NaN,NaN
...,...,...,...,...,...,...,...
256,2025-12-04,6857,23882,3876,6857,23882,3876
257,2025-12-05,6870,24028,3903,6870,24028,3903
258,2025-12-08,6847,24046,3924,6847,24046,3924
259,2025-12-09,6841,24163,3910,6841,24163,3910


In [15]:
for index_name in since_election_without_today.columns[1:]:

    df_to_check[index_name+"overlap"]=df_to_check[index_name]/df_to_check[index_name+"check"]
df_to_check

,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck,S&P500overlap,DAX40overlap,SHANGHAIoverlap
0,2024-12-10,6035,20329,3423,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-12-11,6084,20399,3432,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-12-12,6051,20426,3462,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-12-13,6051,20406,3392,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-12-16,6074,20314,3386,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
256,2025-12-04,6857,23882,3876,6857,23882,3876,1,1,1
257,2025-12-05,6870,24028,3903,6870,24028,3903,1,1,1
258,2025-12-08,6847,24046,3924,6847,24046,3924,1,1,1
259,2025-12-09,6841,24163,3910,6841,24163,3910,1,1,1


In [16]:
overlap_cols=df_to_check.columns[df_to_check.columns.str.contains("overlap")]
overlap_cols

Index(['S&P500overlap', 'DAX40overlap', 'SHANGHAIoverlap'], dtype='object')

In [17]:
(df_to_check[overlap_cols]==1).all(axis=1)

0      False
1      False
2      False
3      False
4      False
       ...  
256     True
257     True
258     True
259     True
260    False
Length: 261, dtype: bool

In [18]:
df_to_check["all_equal"]=(df_to_check[overlap_cols]==1).all(axis=1)
df_to_check

,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck,S&P500overlap,DAX40overlap,SHANGHAIoverlap,all_equal
0,2024-12-10,6035,20329,3423,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2024-12-11,6084,20399,3432,NaN,NaN,NaN,NaN,NaN,NaN,False
2,2024-12-12,6051,20426,3462,NaN,NaN,NaN,NaN,NaN,NaN,False
3,2024-12-13,6051,20406,3392,NaN,NaN,NaN,NaN,NaN,NaN,False
4,2024-12-16,6074,20314,3386,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
256,2025-12-04,6857,23882,3876,6857,23882,3876,1,1,1,True
257,2025-12-05,6870,24028,3903,6870,24028,3903,1,1,1,True
258,2025-12-08,6847,24046,3924,6847,24046,3924,1,1,1,True
259,2025-12-09,6841,24163,3910,6841,24163,3910,1,1,1,True


In [19]:

checked_df=df_to_check[df_to_check["all_equal"]==True]
checked_df

,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck,S&P500overlap,DAX40overlap,SHANGHAIoverlap,all_equal
238,2025-11-10,6832,23960,4019,6832,23960,4019,1,1,1,True
239,2025-11-11,6847,24088,4003,6847,24088,4003,1,1,1,True
240,2025-11-12,6851,24381,4000,6851,24381,4000,1,1,1,True
241,2025-11-13,6737,24042,4030,6737,24042,4030,1,1,1,True
242,2025-11-14,6734,23877,3990,6734,23877,3990,1,1,1,True
243,2025-11-17,6672,23591,3972,6672,23591,3972,1,1,1,True
244,2025-11-18,6617,23181,3940,6617,23181,3940,1,1,1,True
245,2025-11-19,6642,23163,3947,6642,23163,3947,1,1,1,True
246,2025-11-20,6539,23279,3931,6539,23279,3931,1,1,1,True
247,2025-11-21,6603,23092,3835,6603,23092,3835,1,1,1,True


In [20]:
checked_df_red=checked_df[checked_df.columns[:4]]
checked_df_red["Date"]=checked_df_red["Date"].dt.strftime("%Y-%m-%d")
checked_df_red

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_85003/3470133631.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  checked_df_red["Date"]=checked_df_red["Date"].dt.strftime("%Y-%m-%d")


,Date,S&P500,DAX40,SHANGHAI
238,2025-11-10,6832,23960,4019
239,2025-11-11,6847,24088,4003
240,2025-11-12,6851,24381,4000
241,2025-11-13,6737,24042,4030
242,2025-11-14,6734,23877,3990
243,2025-11-17,6672,23591,3972
244,2025-11-18,6617,23181,3940
245,2025-11-19,6642,23163,3947
246,2025-11-20,6539,23279,3931
247,2025-11-21,6603,23092,3835


In [21]:
#Open the google spreadsheet
sh = gc.open('TrumptrackerRaw')
sheet_name = "RawIndices"  # Replace with the name of the sheet you want to select
wks = sh.worksheet_by_title(sheet_name)


#Get the data from the Sheet into python as DF
read_indices = wks.get_as_df()
read_indices_red=read_indices[read_indices.columns[:4]]
read_indices_red

,Date,S&P500,DAX40,SHANGHAI
0,2024-12-10,6035,20329,3423
1,2024-12-11,6084,20399,3432
2,2024-12-12,6051,20426,3462
3,2024-12-13,6051,20406,3392
4,2024-12-16,6074,20314,3386
...,...,...,...,...
255,2025-12-03,6850,23694,3878
256,2025-12-04,6857,23882,3876
257,2025-12-05,6870,24028,3903
258,2025-12-08,6847,24046,3924


In [22]:
read_plus_new_checked=pd.concat([read_indices_red,checked_df_red], axis=0, ignore_index=True)

read_plus_new_checked.drop_duplicates(subset=["Date"], keep="first", inplace=True)
read_plus_new_checked.reset_index(drop=True, inplace=True)
read_plus_new_checked

,Date,S&P500,DAX40,SHANGHAI
0,2024-12-10,6035,20329,3423
1,2024-12-11,6084,20399,3432
2,2024-12-12,6051,20426,3462
3,2024-12-13,6051,20406,3392
4,2024-12-16,6074,20314,3386
...,...,...,...,...
255,2025-12-03,6850,23694,3878
256,2025-12-04,6857,23882,3876
257,2025-12-05,6870,24028,3903
258,2025-12-08,6847,24046,3924


In [23]:
read_plus_new_checked["Date"]=pd.to_datetime(read_plus_new_checked["Date"])
read_plus_new_checked=read_plus_new_checked[read_plus_new_checked["Date"]>date_365_days_ago]
read_plus_new_checked

,Date,S&P500,DAX40,SHANGHAI
1,2024-12-11,6084,20399,3432
2,2024-12-12,6051,20426,3462
3,2024-12-13,6051,20406,3392
4,2024-12-16,6074,20314,3386
5,2024-12-17,6051,20246,3361
...,...,...,...,...
255,2025-12-03,6850,23694,3878
256,2025-12-04,6857,23882,3876
257,2025-12-05,6870,24028,3903
258,2025-12-08,6847,24046,3924


In [24]:
def converter_perc(index_name):
    df_perc=pd.DataFrame()
    df_perc["Date"]=read_plus_new_checked["Date"]
    col_important=read_plus_new_checked[index_name]
    base_value=col_important.iloc[0]
    df_perc[index_name+"_perc"]=(col_important-base_value)*100/base_value
    df_perc[index_name+"_perc"] = df_perc[index_name+"_perc"].apply(lambda x: float(Decimal(str(x)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)))

    return(df_perc)

In [25]:
for col in read_plus_new_checked.columns[1:]:
    df_perc=converter_perc(col)
    read_plus_new_checked=pd.merge(read_plus_new_checked,df_perc,on="Date",how="outer",suffixes=["_abs","_perc"])
read_plus_new_checked 

,Date,S&P500,DAX40,SHANGHAI,S&P500_perc,DAX40_perc,SHANGHAI_perc
0,2024-12-11,6084,20399,3432,0.00,0.00,0.00
1,2024-12-12,6051,20426,3462,-0.54,0.13,0.87
2,2024-12-13,6051,20406,3392,-0.54,0.03,-1.17
3,2024-12-16,6074,20314,3386,-0.16,-0.42,-1.34
4,2024-12-17,6051,20246,3361,-0.54,-0.75,-2.07
...,...,...,...,...,...,...,...
254,2025-12-03,6850,23694,3878,12.59,16.15,13.00
255,2025-12-04,6857,23882,3876,12.71,17.07,12.94
256,2025-12-05,6870,24028,3903,12.92,17.79,13.72
257,2025-12-08,6847,24046,3924,12.54,17.88,14.34


In [26]:



#open the google spreadsheet (where 'PY to Gsheet Test' is the name of my sheet)
sh = gc.open('TrumptrackerRaw')

#select the first sheet 
wks = sh[0]

wks.set_dataframe(read_plus_new_checked,(0,0))


/Users/halukamaier-borst/Documents/CodeRelated/HBDataviz/2025/08_Trumptracker/.venv/lib/python3.12/site-packages/pygsheets/worksheet.py:1431: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(nan)


In [27]:
# Get the current year
current_year = datetime.now().year
print(current_year)

2025


In [28]:
df=pd.DataFrame()
for year in range(2024,current_year+1):
    print(year)
    url=f"https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/{year}/all?type=daily_treasury_yield_curve&field_tdr_date_value={year}&page&_format=csv"
    response = requests.get(url)

    file_object = io.StringIO(response.content.decode('utf-8'))
    data=pd.read_csv(file_object)
    df=pd.concat([df,data],axis=0)
df

2024
2025


,Date,1 Mo,2 Mo,3 Mo,4 Mo,6 Mo,1 Yr,2 Yr,3 Yr,5 Yr,7 Yr,10 Yr,20 Yr,30 Yr,1.5 Month
0,12/31/2024,4.40,4.39,4.37,4.32,4.24,4.16,4.25,4.27,4.38,4.48,4.58,4.86,4.78,NaN
1,12/30/2024,4.43,4.42,4.37,4.33,4.25,4.17,4.24,4.29,4.37,4.46,4.55,4.84,4.77,NaN
2,12/27/2024,4.44,4.43,4.31,4.35,4.29,4.20,4.31,4.36,4.45,4.53,4.62,4.89,4.82,NaN
3,12/26/2024,4.45,4.45,4.35,4.37,4.31,4.23,4.30,4.35,4.42,4.49,4.58,4.83,4.76,NaN
4,12/24/2024,4.44,4.44,4.40,4.38,4.30,4.24,4.29,4.36,4.43,4.52,4.59,4.84,4.76,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,01/08/2025,4.41,4.34,4.35,4.31,4.25,4.19,4.28,4.31,4.45,4.56,4.67,4.97,4.91,NaN
230,01/07/2025,4.42,4.35,4.35,4.31,4.24,4.19,4.30,4.33,4.46,4.57,4.67,4.97,4.91,NaN
231,01/06/2025,4.43,4.36,4.35,4.31,4.24,4.17,4.28,4.30,4.42,4.52,4.62,4.91,4.85,NaN
232,01/03/2025,4.44,4.35,4.34,4.31,4.25,4.18,4.28,4.32,4.41,4.51,4.60,4.88,4.82,NaN


In [29]:
df["Date"]=pd.to_datetime(df["Date"])
since_election_yield=df[df["Date"]>=date_365_days_ago]
since_election_yield=since_election_yield[["Date","10 Yr"]]
since_election_yield=since_election_yield.sort_values(by="Date", ascending=True)
since_election_yield

,Date,10 Yr
13,2024-12-11,4.26
12,2024-12-12,4.32
11,2024-12-13,4.40
10,2024-12-16,4.39
9,2024-12-17,4.40
...,...,...
4,2025-12-03,4.06
3,2025-12-04,4.11
2,2025-12-05,4.14
1,2025-12-08,4.17


In [30]:



#open the google spreadsheet (where 'PY to Gsheet Test' is the name of my sheet)
sh = gc.open('TrumptrackerRaw')

#select the first sheet 
sheet_name = "Yield"  # Replace with the name of the sheet you want to select
wks = sh.worksheet_by_title(sheet_name)


wks.set_dataframe(since_election_yield,(0,0))


In [31]:
if "DWKEY" in os.environ:

    DWKEY = os.environ["DWKEY"]
    print(len(DWKEY))


else:
    data=pd.read_csv("/Users/halukamaier-borst/Documents/Keys.csv")
    DWKEY=data.APIKey.iloc[1]

In [32]:
dw_connection = datawrapper.Datawrapper(access_token = DWKEY)

allcharts=dw_connection.get_charts(limit=500)

dw_connection.publish_chart(chart_id="50HqS")

{'data': {'publicId': '50HqS',
  'language': 'de-DE',
  'theme': 'handelsblatt',
  'id': '50HqS',
  'type': 'd3-lines',
  'title': 'So haben sich die Märkte in den letzten 365 Tagen verändert',
  'lastEditStep': 5,
  'publishedAt': '2025-12-10T08:48:11.992Z',
  'publicUrl': 'https://datawrapper.dwcdn.net/50HqS/827/',
  'publicVersion': 827,
  'deleted': False,
  'forkable': False,
  'isFork': False,
  'metadata': {'data': {'changes': [],
    'transpose': False,
    'vertical-header': True,
    'horizontal-header': True,
    'column-format': {'DAX40': {'type': 'auto', 'ignore': True},
     'S&P500': {'type': 'auto', 'ignore': True},
     'SHANGHAI': {'type': 'auto', 'ignore': True},
     'DAX40_abs': {'type': 'auto', 'ignore': True},
     'DAX40_perc': {'type': 'auto',
      'number-append': ' %',
      'number-prepend': ''},
     'S&P500_abs': {'type': 'auto', 'ignore': True},
     'S&P500_perc': {'type': 'auto',
      'number-append': ' %',
      'number-prepend': ''},
     'SHANGHAI_